# Paper 3 — CTU-13 bounded warning pilot

**CPU · resume enabled · 25 LR fits · 175 external evaluations**

Use **Runtime → Run all**. Approve Drive access. A separate Python environment avoids mixed dependency versions; no kernel restart is needed. This notebook reuses the completed qualification panels and writes only to a new pilot folder. Keep the original inventory and qualification folders unchanged.

This is a post hoc supplementary pilot with five source captures from one institution. It does not measure production prevalence or establish general warning utility. Full-data runtime has not yet been measured. Allow 1–2 GB extra Drive space.


In [ ]:
from google.colab import drive
from pathlib import Path
import sys, os, json, subprocess, zipfile, hashlib, base64, io, tempfile

drive.mount('/content/drive')
PAPER3_ROOT = Path('/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet')
PARENT = PAPER3_ROOT / 'supplementary_ctu13_partition_qualification_20261027' / 'run_v1'
if not (PARENT / 'summary.json').is_file():
    raise FileNotFoundError('Completed qualification folder not found: ' + str(PARENT))
if json.loads((PARENT / 'summary.json').read_text())['status'] != 'CTU13_PARTITION_QUALIFICATION_COMPLETE':
    raise RuntimeError('The qualification stage must finish successfully before this pilot.')
print('Qualification folder found:', PARENT)


## Prepare the embedded workflow

The payload hash is checked before extraction. The embedded plan and input pins are fixed for this notebook.


In [ ]:
PAYLOAD_B64 = '...'
PAYLOAD_SHA256 = '287972da3c48a756b018ebe3920227264d60e3bb614e647cb2bdcc0ff70f72dd'
payload = base64.b64decode(PAYLOAD_B64)
if hashlib.sha256(payload).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError('Embedded payload differs from the prepared notebook.')
PACKAGE = Path('/content/paper3_ctu_warning_pilot_package')
PACKAGE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for n in z.namelist():
        if Path(n).is_absolute() or '..' in Path(n).parts:
            raise RuntimeError('Unexpected payload path')
    z.extractall(PACKAGE)
print('Prepared:', PACKAGE)


## Isolated dependencies

This installs the tested package versions in a separate environment. It does not replace packages already loaded in the Colab notebook kernel. Installation may take several minutes and requires internet access.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess

ENV = Path('/content/paper3_ctu_warning_pilot_env')
PYTHON = ENV / 'bin' / 'python'
VERSIONS = {'numpy':'2.3.5', 'scipy':'1.17.0', 'scikit-learn':'1.8.0', 'joblib':'1.5.3', 'threadpoolctl':'3.6.0'}

# Colab may not provide ensurepip. Also repairs an incomplete environment
# left by the earlier setup attempt, without clearing installed packages.
created = subprocess.run(
    [sys.executable, '-m', 'venv', '--without-pip', str(ENV)],
    text=True, capture_output=True
)
if created.returncode != 0:
    print(created.stdout)
    print(created.stderr)
    raise RuntimeError('Environment creation failed. Send the full message printed above.')

probe = "import importlib.metadata as m,json; print(json.dumps({n:m.version(n) for n in " + repr(list(VERSIONS)) + "}))"
check = subprocess.run([str(PYTHON), '-c', probe], text=True, capture_output=True)
ready = check.returncode == 0 and json.loads(check.stdout) == VERSIONS
if not ready:
    # Run Colab's existing pip against the isolated interpreter; no ensurepip needed.
    command = [sys.executable, '-m', 'pip', '--python', str(PYTHON),
               'install', '--disable-pip-version-check', '--only-binary=:all:']
    command += [n+'=='+v for n,v in VERSIONS.items()]
    subprocess.run(command, check=True)
check = subprocess.run([str(PYTHON), '-c', probe], text=True, capture_output=True, check=True)
if json.loads(check.stdout) != VERSIONS:
    raise RuntimeError('Isolated dependency versions do not match.')
RUN_ENV = dict(os.environ)
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    RUN_ENV[key] = '1'
print('Isolated dependencies ready:', check.stdout.strip())


## Run or resume

Progress is printed for every warning group and completed source unit. Completed checkpoints are reused after verification. If interrupted, run this same notebook again. An error produces a diagnostic report when possible. No target adaptation or automatic threshold relaxation occurs.


In [ ]:
REPORT_ZIP = None
command = [str(PYTHON), '-u', str(PACKAGE / 'engine.py'), '--root', str(PAPER3_ROOT)]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=RUN_ENV) as proc:
    for line in proc.stdout:
        print(line, end='', flush=True)
        if line.startswith('REPORT_ZIP='):
            REPORT_ZIP = Path(line.strip().split('=',1)[1])
    returncode = proc.wait()
if returncode != 0 or REPORT_ZIP is None or not REPORT_ZIP.is_file():
    raise RuntimeError('Pilot initialization or execution was blocked. Keep the error text above; do not edit checkpoints.')
print('Report ready:', REPORT_ZIP)


## Verify and download the report

The saved-result verifier checks hashes, metrics, threshold choice, rank correlations and strict regression calculations. It does not independently repeat raw recovery or predictions from the omitted input features. Return the downloaded ZIP for review before updating manuscript claims.


In [ ]:
with tempfile.TemporaryDirectory(prefix='ctu_warning_report_') as tmp:
    with zipfile.ZipFile(REPORT_ZIP) as z:
        for n in z.namelist():
            if Path(n).is_absolute() or '..' in Path(n).parts:
                raise RuntimeError('Unexpected report path')
        z.extractall(tmp)
    summary = json.loads((Path(tmp) / 'summary.json').read_text())
    print('Experiment status:', summary['status'])
    if summary['status'] == 'CTU13_WARNING_PILOT_COMPLETE':
        subprocess.run([str(PYTHON), str(PACKAGE / 'verify_report.py'), tmp], check=True, env=RUN_ENV)
    else:
        print('Run is partial. Return this diagnostic ZIP before changing anything:')
        print(json.dumps(summary['errors'], indent=2))
from google.colab import files
files.download(str(REPORT_ZIP))
print('If the browser download is blocked, the same ZIP was saved beside run_v1 in the new Drive pilot folder.')
